# Exercice 02 — Multiplication de matrices

On programme le produit matriciel **à la main** avec trois boucles, puis on compare avec **NumPy**.

Si $A$ a la taille $n\times m$ et $B$ la taille $p\times q$, alors $AB$ existe seulement si $m=p$. Le résultat mesure $n\times q$.

$$C_{ij}=\sum_{k=1}^{m} A_{ik}B_{kj}$$

Pour commencer, essayons :

$$A=\begin{pmatrix}1&2&3\\4&5&6\end{pmatrix},\quad B=\begin{pmatrix}7&8\\9&10\\11&12\end{pmatrix}$$

Le résultat attendu est $\begin{pmatrix}58&64\\139&154\end{pmatrix}$.

**Modes :** `1` pour multiplier une matrice carrée par elle-même (`A × A`), `2` pour saisir deux matrices (`A × B`).

In [ ]:
import math

# ============================================================
# EXERCICE 02 — MULTIPLICATION MATRICIELLE EN PYTHON
# ============================================================
# THEORIE
# Si A est n x m et B est p x q, il faut m == p.
# Le resultat C est n x q, avec :
#      C[i][j] = somme(A[i][k] * B[k][j]) pour k = 0..m-1.
# C'est un PRODUIT LIGNE-COLONNE, PAS un produit case par case.
# En general, A @ B est DIFFERENT de B @ A (parfois B @ A
# n'est meme pas defini).
#
# VIE QUOTIDIENNE
# Une matrice de quantites (magasins x produits), multipliee
# par un vecteur de prix (produits x 1), donne une depense ou
# une recette totale par magasin (magasins x 1).
#
# INFORMATIQUE
# Une matrice 2x2 peut representer une rotation ou une mise
# a l'echelle de coordonnees 2D : transformation @ vecteur.
# En 3D, on utilise des matrices plus grandes.
#
# INTELLIGENCE ARTIFICIELLE
# Une couche de reseau de neurones calcule souvent W @ X + b :
# W et X sont multiplies matriciellement ; b est ensuite ajoute.
#
# ARCHITECTURE ET GENIE CIVIL
# Les transformations matricielles changent de repere les
# coordonnees d'un plan. En calcul des structures, K @ u = F
# lie rigidites, deplacements et forces dans un modele.
# Ce n'est pas a lui seul un calcul de dimensionnement.
# ============================================================


def lire_entier(message, minimum=1, maximum=None):
    while True:
        try:
            valeur = int(input(message))
            if valeur < minimum or (maximum is not None and valeur > maximum):
                print(f"Entrez un entier entre {minimum} et {maximum or 'plus' }.")
                continue
            return valeur
        except ValueError:
            print("Veuillez entrer un entier valide.")


def saisir_matrice(nom, n, m):
    print(f"\nSaisie de {nom} ({n} x {m})")
    matrice = []
    for i in range(n):
        while True:
            try:
                valeurs = [float(v) for v in input(
                    f"Ligne {i + 1} ({m} nombres separes par des espaces) : "
                ).split()]
                if len(valeurs) != m or not all(math.isfinite(v) for v in valeurs):
                    print(f"Entrez exactement {m} nombres finis.")
                    continue
                matrice.append(valeurs)
                break
            except ValueError:
                print("Valeurs incorrectes. Recommencez.")
    return matrice


def dimensions(M):
    if not M or not M[0] or any(len(ligne) != len(M[0]) for ligne in M):
        raise ValueError("La matrice est vide ou ses lignes sont irregulieres.")
    return len(M), len(M[0])


def produit_matriciel(A, B):
    n, m = dimensions(A)
    p, q = dimensions(B)
    if m != p:
        raise ValueError(
            f"Dimensions incompatibles : A={n}x{m}, B={p}x{q}. "
            f"Il faut {m}={p}."
        )

    C = [[0.0] * q for _ in range(n)]

    for i in range(n):             # ligne i de A
        for j in range(q):         # colonne j de B
            for k in range(m):     # produits puis somme ligne-colonne
                # Ex. : C[0][0] = 1*7 + 2*9 + 3*11 = 58
                C[i][j] += A[i][k] * B[k][j]
            if not math.isfinite(C[i][j]):
                raise OverflowError("Resultat numerique non fini.")
    return C


def afficher_matrice(nom, M):
    print(f"\n{nom} =")
    for ligne in M:
        print(ligne)


In [ ]:
# ============================================================
# SAISIE INTERACTIVE
# Mode 1 : une matrice carree A -> calculer A @ A.
# Mode 2 : deux matrices compatibles A et B -> calculer A @ B.
# ============================================================

print("=== MULTIPLICATION DE MATRICES ===")
mode = lire_entier("Mode (1 = A x A, 2 = A x B) : ", minimum=1, maximum=2)

while True:
    n = lire_entier("Nombre de lignes de A : ")
    m = lire_entier("Nombre de colonnes de A : ")
    if mode == 1 and n != m:
        print("Pour A x A, A doit etre carree. Recommencez.")
    else:
        break

A = saisir_matrice("A", n, m)

if mode == 1:
    B = [ligne.copy() for ligne in A]
else:
    while True:
        p = lire_entier("Nombre de lignes de B : ")
        if p == m:
            break
        print(f"Impossible : A a {m} colonnes, B doit avoir {m} lignes.")
    q = lire_entier("Nombre de colonnes de B : ")
    B = saisir_matrice("B", p, q)

C = produit_matriciel(A, B)
afficher_matrice("A", A)
afficher_matrice("B", B)
afficher_matrice("C = A x B", C)
print("Dimensions du resultat :", dimensions(C))

In [ ]:
# ============================================================
# VERIFICATION : LE RESULTAT MANUEL == LE RESULTAT NUMPY
# L'operateur @ realise un PRODUIT MATRICIEL (et non A * B).
# ============================================================
import numpy as np

reference = np.array(A) @ np.array(B)
np.testing.assert_allclose(np.array(C), reference, rtol=1e-9, atol=1e-9)
print("Verification NumPy : OK")
print(reference)

In [ ]:
# ============================================================
# TESTS AUTOMATIQUES — AUCUNE SAISIE NECESSAIRE
# ============================================================
A_test = [[1, 2, 3], [4, 5, 6]]
B_test = [[7, 8], [9, 10], [11, 12]]
assert produit_matriciel(A_test, B_test) == [[58, 64], [139, 154]]
assert produit_matriciel([[1, 2], [3, 4]], [[1, 2], [3, 4]]) == [[7, 10], [15, 22]]
try:
    produit_matriciel([[1, 2, 3]], [[1, 2]])
    raise AssertionError("Un produit incompatible aurait du etre refuse.")
except ValueError:
    pass
print("Tests Python : OK (rectangulaire, A x A, dimensions invalides)")

### Exercice personnel

1. Essaie une matrice A de taille **2 × 3** et B de taille **3 × 1**.
2. Essaie le mode `A × A` avec une matrice carrée **3 × 3**.
3. Pourquoi une matrice **2 × 3** ne peut-elle pas être multipliée par une matrice **2 × 2** dans cet ordre ?
4. Compare `A @ B` et `B @ A` avec deux matrices carrées : sont-ils toujours égaux ?